In [ ]:
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import TCT

import matplotlib.pyplot as plt
import requests

In [ ]:
from TCT.translator_resources import TranslatorResources
resources = TranslatorResources.load()

In [ ]:
# Example: Add a custom TRAPI API by fetching its meta_knowledge_graph
# Here we demonstrate with a Pharmacogenomics KP endpoint, registered under a custom name
custom_api_name = "My Pharmacogenomics KP"
custom_query_url = "https://multiomics.rtx.ai:9990/PharmacogenomicsKG/query"
meta_kg_url = "https://multiomics.rtx.ai:9990/PharmacogenomicsKG/meta_knowledge_graph"

response = requests.get(meta_kg_url)
response.raise_for_status()
data = response.json()
for edge in data["edges"]:
    resources.api_names, resources.meta_kg = translator_metakg.add_new_API_for_query(
        resources.api_names, resources.meta_kg, custom_api_name, custom_query_url,
        edge['predicate'], edge['subject'], edge['object'])
resources.rebuild_predicates()
print(f"Added {custom_api_name} with {len(data['edges'])} edges")

In [ ]:
import networkx as nx
selected_KGs = [custom_api_name]

metaKG_sele = resources.meta_kg[resources.meta_kg['API'].isin(selected_KGs)]

# build a multigraph to capture all edges (including duplicates) and their predicates
G = nx.MultiGraph()
for _, row in metaKG_sele.iterrows():
        G.add_edge(row['Subject'], row['Object'], predicate=row['Predicate'])

# layout and draw nodes + edges
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5, iterations=20)
nx.draw(G, pos,
                with_labels=True,
                node_size=50,
                font_size=12,
                font_color='black',
                node_color='blue',
                edge_color='gray')

# draw edge labels
edge_labels = nx.get_edge_attributes(G, 'predicate')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title(f'Interaction Graph: {custom_api_name}')
plt.show()

In [ ]:
metaKG_sele

In [ ]:
# select a list of APIs to use and a list of predicates to use
selected_APIlist = [
    custom_api_name,
    'Clinical Trials KP - TRAPI 1.5.0', 
    'Drug Approvals KP - TRAPI 1.5.0',
]

filtered = resources.filter(selected_APIlist) if selected_APIlist else resources
print(filtered.api_names)
print(filtered.meta_kg.shape)

In [ ]:
nb_result = TCT.Neiborhood_finder(input_node='MONDO:0018874',  # AML (Acute Myeloid Leukemia)
                                   node2_categories=['biolink:SmallMolecule', 'biolink:Drug', 'biolink:ChemicalEntity'],
                                   resources=filtered)
input_node_id = nb_result.input_node_id
result = nb_result.knowledge_graph
result_parsed = nb_result.parsed
result_ranked_by_primary_infores = nb_result.ranked

In [ ]:
# Step 8: Visualize the results
TCT.visulization_one_hop_ranking(result_ranked_by_primary_infores=result_ranked_by_primary_infores,
                                result_parsed=result_parsed,
                                num_of_nodes=50,
                                input_query=input_node_id,
                                fontsize=5)

In [ ]:
path_result = TCT.Path_finder(input_node1='MONDO:0018874',  # AML
                               input_node2='NCBIGene:7157',  # TP53
                               intermediate_categories=['biolink:Drug', 'biolink:SmallMolecule', 'biolink:ChemicalEntity'],
                               resources=resources)
paths = path_result.paths
input_node1_id = path_result.node1_id
input_node2_id = path_result.node2_id
result1 = path_result.knowledge_graph1
result2 = path_result.knowledge_graph2
result_parsed1 = path_result.parsed1
result_parsed2 = path_result.parsed2
result_ranked_by_primary_infores1 = path_result.ranked1
result_ranked_by_primary_infores2 = path_result.ranked2

In [ ]:
paths.head(20)

In [ ]:
forplot = TCT.visulize_path(input_node1_id=input_node1_id,
                            intermediate_node=name_resolver.lookup('Azacitidine').curie,
                            input_node3_id=input_node2_id,
                            result=result1,
                            result2=result2) 